# Prism Race v2: The Mod Wheel
**Runtime → A100 or T4**

From race v1: Prism-Taper reached baseline quality in 4.8x fewer steps.
Can we push further with continuous spectral modulation?

Five contestants:
1. **Baseline**: Standard init
2. **Prism-Taper**: Half LR (4.8x winner from v1)
3. **Prism-Mod-Gentle**: Taper + gentle mod wheel (0.005, decay 0.999)
4. **Prism-Mod-Strong**: Taper + strong mod wheel (0.02, decay 0.998)
5. **Prism-Mod-Sustained**: Taper + sustained mod (0.01, decay 0.9999)

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py
print('Ready.')

In [ ]:
# Cell 2: Train teacher + extract spectra (skip if cached)
import os, subprocess
if os.path.exists('.prism_cache/shakespeare/spectra.json'):
    print('Spectra cached. Skipping.')
else:
    print('Training teacher (2K steps)...')
    subprocess.run(['python', 'train.py', 'config/train_shakespeare_char.py',
        '--max_iters=2000', '--eval_interval=1000', '--eval_iters=50',
        '--log_interval=1000', '--out_dir=out-teacher',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False'], timeout=600)
    print('Extracting spectra...')
    subprocess.run(['python', 'prism_extract.py',
        '--ckpt', 'out-teacher/ckpt.pt',
        '--out', '.prism_cache/shakespeare'], timeout=120)
    print('Done.')

In [ ]:
# Cell 3: Run all five contestants
import subprocess, time, re, json

STEPS = 5000
EVAL = 100

PRISM_ARGS = [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
]

configs = [
    ('baseline', ['--prism_init=False']),
    ('prism_taper', PRISM_ARGS + ['--learning_rate=5e-4', '--warmup_iters=50', '--prism_mod=0.0']),
    ('mod_gentle', PRISM_ARGS + ['--learning_rate=5e-4', '--warmup_iters=50', '--prism_mod=0.005', '--prism_mod_decay=0.999']),
    ('mod_strong', PRISM_ARGS + ['--learning_rate=5e-4', '--warmup_iters=50', '--prism_mod=0.02', '--prism_mod_decay=0.998']),
    ('mod_sustained', PRISM_ARGS + ['--learning_rate=5e-4', '--warmup_iters=50', '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
]

all_curves = {}
all_meta = {}

for name, extra in configs:
    log_path = f'/content/race2_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP — already done')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'\n{"="*60}')
        print(f'  {name}')
        print(f'{"="*60}')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=100',
             f'--out_dir=out-race2-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        wall = time.time() - t0
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'ERROR: {result.stderr[-500:]}')
        print(f'  Wall time: {wall:.0f}s')

    # Parse val losses
    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    best_v = min(val.values()) if val else 999
    best_s = min(val, key=val.get) if val else 0
    all_meta[name] = {'best_val': best_v, 'best_step': best_s}
    print(f'  [{name}] best val: {best_v:.4f} at step {best_s}')

# Save
with open('/content/race2_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 4: Race results
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/race2_curves.json')).items()}

# Target: baseline's best
bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*70)
print(f'  TARGET: val_loss ≤ {bb:.4f} (baseline best at step {bs})')
print('='*70)

# Time to target for each
print(f'\n{"Runner":>15s}  {"Steps":>7s}  {"Speedup":>8s}  {"Best val":>10s}  {"Best@":>6s}')
print('-' * 55)
for name in ['baseline', 'prism_taper', 'mod_gentle', 'mod_strong', 'mod_sustained']:
    c = curves[name]
    best_v = min(c.values())
    best_s = min(c, key=c.get)
    hit = None
    for s in sorted(c.keys()):
        if c[s] <= bb:
            hit = s
            break
    if hit:
        print(f'{name:>15s}  {hit:>7d}  {bs/hit:>7.1f}x  {best_v:>10.4f}  {best_s:>6d}')
    else:
        print(f'{name:>15s}  {"never":>7s}  {"—":>8s}  {best_v:>10.4f}  {best_s:>6d}')

# Key comparison steps
print(f'\n--- Val loss at key steps ---')
print(f'{"Step":>6s}', end='')
for name in curves: print(f'  {name:>12s}', end='')
print()
for step in [0, 100, 200, 300, 400, 500, 600, 800, 1000, 1500, 2000, 3000, 5000]:
    vals = {n: curves[n].get(step) for n in curves}
    if any(vals.values()):
        print(f'{step:>6d}', end='')
        best = min((v for v in vals.values() if v), default=999)
        for name in curves:
            v = vals[name]
            if v:
                marker = ' *' if abs(v - best) < 0.001 else '  '
                print(f'  {v:>10.4f}{marker}', end='')
            else:
                print(f'  {"":>12s}', end='')
        print()

In [ ]:
# Cell 5: Download
from google.colab import files
for f in ['/content/race2_curves.json'] + [f'/content/race2_{n}.txt' for n in ['baseline','prism_taper','mod_gentle','mod_strong','mod_sustained']]:
    try: files.download(f)
    except: pass